# Team 3: Attention Mechanism Efficiency Profiling on NVIDIA Tesla T4
**Benchmarking Naive Softmax vs SDPA (Memory-Efficient) vs Sparse vs Linear Attention**

This notebook profiles execution latency, peak VRAM consumption, and empirical scaling exponents ($lpha$) across sequence lengths $N \in [64, \dots, 65536]$ on Tesla T4.

In [ ]:
import os, sys, math, time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

print('PyTorch Version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device Name:', torch.cuda.get_device_name(0))

## 1. Attention Implementations
Under our controlled layer setup ($B=1, H=4, d_k=64, d_v=64$):

In [ ]:
def softmax_attention(Q, K, V):
    scale = 1.0 / math.sqrt(Q.shape[-1])
    scores = (Q @ K.transpose(-1, -2)) * scale
    weights = torch.softmax(scores, dim=-1)
    return weights @ V

def linear_attention(Q, K, V):
    # Feature map phi(x) = ReLU(x) + 1.0
    fq = F.relu(Q) + 1.0
    fk = F.relu(K) + 1.0
    acc = torch.float32 if Q.dtype in (torch.float16, torch.bfloat16) else Q.dtype
    fq, fk = fq.to(acc), fk.to(acc)
    kv = fk.transpose(-1, -2) @ V.to(acc)            # (B, H, r, dv)
    z = fk.sum(-2, keepdim=True).transpose(-1, -2)   # (B, H, r, 1)
    num = fq @ kv
    den = (fq @ z).clamp_min(1e-6)
    return (num / den).to(V.dtype)

## 2. Load Empirical Tesla T4 Profiling Data
Data ingested directly from `data/efficiency_raw.csv`.

In [ ]:
df_eff = pd.read_csv('../data/efficiency_raw.csv')
df_eff.head(15)

## 3. Scaling Exponent Fitting ($lpha$) & Speedup Comparison

In [ ]:
ok_df = df_eff[(df_eff.status == 'ok') & (df_eff.N >= 1024)]
print('Log-log Scaling Slopes (Latency vs N):')
for method, grp in ok_df.groupby('method'):
    slope, intercept = np.polyfit(np.log(grp.N), np.log(grp.median_ms), 1)
    print(f'  {method:25s}: alpha = {slope:.3f}')

## 4. Visualizing Latency & Peak VRAM Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for method, grp in df_eff[df_eff.status == 'ok'].groupby('method'):
    axes[0].plot(grp.N, grp.median_ms, marker='o', label=method)
axes[0].set_xscale('log')
axes[0].set_yscale('log')
axes[0].set_title('Latency vs Sequence Length N (log-log)')
axes[0].set_xlabel('Sequence Length N')
axes[0].set_ylabel('Median Execution Time (ms)')
axes[0].grid(True, which='both', ls='--', alpha=0.5)
axes[0].legend()

for method, grp in df_eff[df_eff.status == 'ok'].groupby('method'):
    axes[1].plot(grp.N, grp.extra_mem_mb, marker='s', label=method)
axes[1].set_xscale('log')
axes[1].set_yscale('log')
axes[1].set_title('Peak Intermediate VRAM vs Sequence Length N')
axes[1].set_xlabel('Sequence Length N')
axes[1].set_ylabel('VRAM Allocated (MB)')
axes[1].grid(True, which='both', ls='--', alpha=0.5)
axes[1].legend()

plt.tight_layout()
plt.show()